# HVAC v19 detection retrain — turnkey Colab (NO Google Drive needed)

**Before you start:** `Runtime -> Change runtime type -> GPU` (T4 is fine), then `Runtime -> Run all`.

No Drive / no phone sign-in. Cell 3 asks you to pick the bundle file directly
(`hvac_v19s_tiled_colab.zip`, ~940 MB) from your computer.

At the end it produces `hvac_yolov8s_v19.pt` and downloads it to your computer.
Drop that into the repo as `models/hvac_yolov8s_v19.pt` and tell me — I run the gate
(must beat F1 0.889 on held-out) and deploy if it wins.

In [ ]:
# 1. Confirm GPU is on (pure Python - no shell)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE  ->  Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
# 2. Install YOLO
!pip -q install ultralytics==8.4.51

In [ ]:
# 3. Upload the bundle from your computer (pick hvac_v19s_tiled_colab.zip).
#    ~940 MB - the upload can take several minutes; let it finish.
from google.colab import files
import zipfile, os
up = files.upload()
zname = list(up.keys())[0]
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('unzipped to', os.getcwd())
print('contents:', sorted(os.listdir('.'))[:10])

In [ ]:
# 4. Train (v10 base -> v19). ~25-40 min on a T4. Recipe = the v14-fix: SGD, lr0=0.01, mosaic=0.5.
!python train_v11.py --data-yaml yolo_dataset_v19s_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19

In [ ]:
# 5. Download the trained model to your computer
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output for errors.'
print('downloading', cands[0])
files.download(cands[0])